# **Data analysis on followers**

This notebook runs a bunch of data analysis on the followers and saves the analysis to `data/processed/followers`.

In [ ]:
import sys
import subprocess


subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "pandas",
    "plotly",
    "wordcloud",
    "matplotlib",
    "numpy",
    "kaleido",
    "pyarrow"
])

# Imports and setup

In [21]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
import numpy as np
import matplotlib.pyplot as plt  
from pathlib import Path
import json
import plotly.io as pio

df = pd.read_parquet('../data/project_followers.parquet')
index = {
    "numerical":{},
    "top":{},
    "graphs":{},
}

In [38]:
EXPORT_DIR = Path('../data/processed/followers').resolve()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_WIDTH = 1100
DEFAULT_HEIGHT = 420

som_wide = go.layout.Template(layout=dict(
    width=DEFAULT_WIDTH,
    margin=dict(l=60, r=30, t=60, b=50),
))
pio.templates['som_wide'] = som_wide
pio.templates.default = 'plotly_white+som_wide'

try:
    pio.renderers.default = 'vscode'
except Exception:
    pass

def register_graph(key: str, name: str, description: str, image_file: str):
    index['graphs'][key] = {
        'name': name,
        'description': description,
        'image_file': Path(image_file).name
    }

def save_plotly(fig, name: str, description: str, scale: int = 2, width=None, height=None, friendly_name: str = ""):
    filename = f"{name}.png"
    out_path = EXPORT_DIR / filename
    try:
        exp_width = width if width is not None else (fig.layout.width or DEFAULT_WIDTH)
        exp_height = height if height is not None else (fig.layout.height or DEFAULT_HEIGHT)
        fig.write_image(str(out_path), scale=scale, width=exp_width, height=exp_height)
        register_graph(name, friendly_name, description, filename)
    except Exception as e:
        print(f'Failed to save Plotly figure {name}: {e}')

def save_matplotlib_current(name: str, description: str, dpi: int = 150, friendly_name: str = ""):
    filename = f"{name}.png"
    out_path = EXPORT_DIR / filename
    try:
        plt.gcf().savefig(out_path, bbox_inches='tight', dpi=dpi, facecolor='white')
        register_graph(name, friendly_name, description, filename)
    except Exception as e:
        print(f'Failed to save Matplotlib figure {name}: {e}')

def save_to_index(key: str, name: str, description: str, data, type="numerical"):
    if isinstance(data, pd.DataFrame):
        records = data.to_dict(orient='records')
        indexed = {i: rec for i, rec in enumerate(records)}
        index[type][key] = {
            'name': name,
            'description': description,
            'data': indexed,
        }
    else:
        index[type][key] = {
            'name': name,
            'description': description,
            'data': data
        }

print(f"Followers dataset shape: {df.shape}")

Followers dataset shape: (2882, 8)


In [40]:
users_df = pd.read_parquet('../data/users.parquet')
projects_df = pd.read_parquet('../data/projects.parquet')


df_original = pd.read_parquet('../data/project_followers.parquet')


In [41]:
df = df_original.copy()


users_df = pd.read_parquet('../data/users.parquet')
projects_df = pd.read_parquet('../data/projects.parquet')

df = df.merge(
    projects_df[['id', 'title', 'category', 'user_id']], 
    left_on='project_id', 
    right_on='id', 
    how='left'
).drop('id', axis=1).rename(columns={'title': 'project_title', 'user_id': 'project_owner_id'})

df = df.merge(
    users_df[['id', 'display_name']], 
    left_on='project_owner_id', 
    right_on='id', 
    how='left'
).drop('id', axis=1).rename(columns={'display_name': 'project_owner_name'})

df = df.drop('follower_name', axis=1)  # Remove original
df = df.merge(
    users_df[['id', 'display_name']], 
    left_on='follower_id', 
    right_on='id', 
    how='left'
).drop('id', axis=1).rename(columns={'display_name': 'follower_name'})



# **Numerical data**
Shows some numerical data about followers

### Total follower relationships

Count the total number of follower relationships in the dataset.

In [42]:
total_relationships = len(df)
print(f"Total follower relationships: {total_relationships}")

save_to_index('total_relationships', name='Total follower relationships', 
              description='The total number of follower relationships in the dataset.', 
              data=int(total_relationships), type='numerical')

Total follower relationships: 2882


### Unique followers

Count the number of unique users who follow at least one project.

In [43]:
unique_followers = df['follower_id'].nunique()
print(f"Unique followers: {unique_followers}")

save_to_index('unique_followers', name='Unique followers', 
              description='The number of unique users who follow at least one project.', 
              data=int(unique_followers), type='numerical')

Unique followers: 935


### Unique projects followed

Count the number of unique projects that have at least one follower.

In [44]:
unique_projects_followed = df['project_id'].nunique()
print(f"Unique projects followed: {unique_projects_followed}")

save_to_index('unique_projects_followed', name='Unique projects followed', 
              description='The number of unique projects that have at least one follower.', 
              data=int(unique_projects_followed), type='numerical')

Unique projects followed: 1375


# **Top stuff**

Shows the top stuff related to followers

### Top 3 projects by follower count

Show the top 3 projects with the most followers.

In [45]:
_df = df.copy()
_df['project_title_clean'] = _df['project_title'].astype('object').fillna('(unnamed)').astype(str)
_df['project_owner_name_clean'] = _df['project_owner_name'].astype('object').fillna('(unnamed)').astype(str)

project_follower_counts = _df.groupby(['project_id', 'project_title_clean', 'project_owner_name_clean', 'category']).size().reset_index(name='follower_count')

top3_projects = project_follower_counts.nlargest(3, 'follower_count').copy()
top3_projects['project_link'] = top3_projects['project_id'].apply(
    lambda x: f"https://summer.hackclub.com/projects/{x}")
top3_projects.reset_index(drop=True, inplace=True)

print("Top 3 projects by follower count:")

save_to_index('top_projects_followers', name='Top 3 projects by follower count',
              description='The top 3 projects by number of followers.', data=top3_projects, type='top')
top3_projects

Top 3 projects by follower count:


,project_id,project_title_clean,project_owner_name_clean,category,follower_count,project_link
0,1014,MemeOS – A Web-Based “Operating System” Filled...,Gabriel Pop,Web App,77,https://summer.hackclub.com/projects/1014
1,26,Lunar,Mohit Tiwari,Web App,45,https://summer.hackclub.com/projects/26
2,4482,Wilderlands,Laura,Video Game,30,https://summer.hackclub.com/projects/4482


### Top 3 users by projects followed

Show the top 3 users who follow the most projects.

In [46]:
_df['follower_name_clean'] = _df['follower_name'].astype('object').fillna('(unnamed)').astype(str)

user_follow_counts = _df.groupby(['follower_id', 'follower_name_clean']).size().reset_index(name='projects_followed')

top3_followers = user_follow_counts.nlargest(3, 'projects_followed').copy()
top3_followers['profile_link'] = top3_followers['follower_id'].apply(
    lambda x: f"https://summer.hackclub.com/users/{x}")
top3_followers.reset_index(drop=True, inplace=True)

print("Top 3 users by projects followed:")

save_to_index('top_users_following', name='Top 3 users by projects followed',
              description='The top 3 users by number of projects they follow.', data=top3_followers, type='top')
top3_followers

Top 3 users by projects followed:


,follower_id,follower_name_clean,projects_followed,profile_link
0,5,Neon,120,https://summer.hackclub.com/users/5
1,5396,Abū al-Barāʾ,107,https://summer.hackclub.com/users/5396
2,126,dave9123,74,https://summer.hackclub.com/users/126


### Top 3 categories by follower interest

Show the top 3 project categories with the most followers.

In [47]:
category_follower_counts = _df.groupby('category').size().reset_index(name='total_followers')

top3_categories = category_follower_counts.nlargest(3, 'total_followers').copy()
top3_categories.reset_index(drop=True, inplace=True)

print("Top 3 categories by follower interest:")

save_to_index('top_categories_followers', name='Top 3 categories by follower interest',
              description='The top 3 project categories by total number of followers.', data=top3_categories, type='top')
top3_categories

Top 3 categories by follower interest:


,category,total_followers
0,Web App,1003
1,Something else,609
2,Video Game,383


# **Graphs**

Various visualizations of follower data

### Follower distribution by project category

A bar chart showing follower interest across different project categories.

In [48]:
category_counts = df.groupby('category').size().reset_index(name='follower_count')
category_counts = category_counts.sort_values('follower_count', ascending=False)

fig = px.bar(
    category_counts.head(10),  # Top 10 categories
    x='category',
    y='follower_count',
    title='Follower Distribution by Project Category (Top 10)',
    labels={'category': 'Project Category', 'follower_count': 'Number of Followers'}
)

fig.update_xaxes(tickangle=45)
fig.update_layout(height=500)
fig.show()

save_plotly(fig, 'follower_category_distribution', 'Distribution of followers across different project categories', 
            friendly_name='Follower category distribution')

### Projects followed per user distribution

A histogram showing how many projects users typically follow.

In [50]:
user_follow_counts = df.groupby('follower_id').size().reset_index(name='projects_followed')

fig = px.histogram(
    user_follow_counts,
    x='projects_followed',
    nbins=30,
    title='Distribution of Projects Followed per User',
    labels={'projects_followed': 'Number of Projects Followed', 'count': 'Number of Users'}
)

fig.update_layout(
    xaxis_title='Number of Projects Followed per User',
    yaxis_title='Number of Users',
    height=500
)
fig.show()

save_plotly(fig, 'projects_followed_per_user_distribution', 'Distribution showing how many projects users typically follow', 
            friendly_name='Projects followed per user distribution')

### Top projects and active followers

A combined visualization showing top projects by followers and most active users by projects followed.

In [51]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 10 Projects by Followers', 'Top 10 Most Active Followers'),
    horizontal_spacing=0.1
)

top_projects = project_follower_counts.nlargest(10, 'follower_count')
top_projects = top_projects.merge(
    df[['project_id', 'project_title']].drop_duplicates(), 
    on='project_id', how='left'
)
top_projects['project_title_short'] = top_projects['project_title'].astype(str).str[:20] + '...'

fig.add_trace(
    go.Bar(
        x=top_projects['follower_count'],
        y=top_projects['project_title_short'],
        orientation='h',
        name='Project Followers',
        marker_color='lightblue'
    ),
    row=1, col=1
)

top_users = user_follow_counts.nlargest(10, 'projects_followed')
top_users = top_users.merge(
    df[['follower_id', 'follower_name']].drop_duplicates(), 
    on='follower_id', how='left'
)
top_users['follower_name_short'] = top_users['follower_name'].astype(str).str[:15] + '...'

fig.add_trace(
    go.Bar(
        x=top_users['projects_followed'],
        y=top_users['follower_name_short'],
        orientation='h',
        name='Projects Followed',
        marker_color='lightcoral'
    ),
    row=1, col=2
)

fig.update_layout(
    height=600,
    title_text="Top Projects and Most Active Followers",
    showlegend=False
)

fig.update_xaxes(title_text="Number of Followers", row=1, col=1)
fig.update_xaxes(title_text="Projects Followed", row=1, col=2)
fig.update_yaxes(title_text="Projects", row=1, col=1)
fig.update_yaxes(title_text="Users", row=1, col=2)

fig.show()

save_plotly(fig, 'top_projects_and_followers', 'Combined view of top projects by followers and most active users', 
            friendly_name='Top projects and active followers')

## Export

Save the analysis index to a JSON file.

### Follower statistics export

Export detailed statistics for projects and users including follower counts and project follow counts.

In [ ]:
project_stats = []
project_follower_data = df.groupby(['project_id', 'project_title', 'project_owner_name', 'category']).size().reset_index(name='follower_count')
project_follower_data = project_follower_data.sort_values('follower_count', ascending=False)

for _, row in project_follower_data.iterrows():
    project_stat = {
        "project_id": row['project_id'],
        "project_title": row['project_title'] if pd.notna(row['project_title']) else "(unnamed)",
        "project_owner": row['project_owner_name'] if pd.notna(row['project_owner_name']) else "(unnamed)",
        "category": row['category'] if pd.notna(row['category']) else "(no category)",
        "follower_count": int(row['follower_count'])
    }
    project_stats.append(project_stat)

# User follow statistics
user_stats = []
user_follow_data = df.groupby(['follower_id', 'follower_name']).size().reset_index(name='projects_followed')
user_follow_data = user_follow_data.sort_values('projects_followed', ascending=False)

for _, row in user_follow_data.iterrows():
    user_stat = {
        "user_id": row['follower_id'],
        "user_name": row['follower_name'] if pd.notna(row['follower_name']) else "(unnamed)",
        "projects_followed": int(row['projects_followed'])
    }
    user_stats.append(user_stat)

category_stats = []
category_data = df.groupby('category').size().reset_index(name='total_followers')
category_data = category_data.sort_values('total_followers', ascending=False)

for _, row in category_data.iterrows():
    category_stat = {
        "category": row['category'] if pd.notna(row['category']) else "(no category)",
        "total_followers": int(row['total_followers'])
    }
    category_stats.append(category_stat)

index['project_statistics'] = project_stats
index['user_statistics'] = user_stats
index['category_statistics'] = category_stats

print(f"- {len(project_stats)} project statistics")
print(f"- {len(user_stats)} user statistics") 
print(f"- {len(category_stats)} category statistics")

- 995 project statistics
- 935 user statistics
- 5 category statistics


In [54]:
output_file = EXPORT_DIR / 'index.json'
with open(output_file, 'w') as f:
    json.dump(index, f, indent=2)

print(f"\nAnalysis saved to {output_file}")


Analysis saved to /home/ajayanto/projects/SoM-Analytics/data/processed/followers/index.json
